In [1]:
from fastapi import UploadFile
import polars as pl
import io
import re
from typing import Optional, Sequence
from dataclasses import dataclass, field
import chess
from chess import pgn
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
def create_uploadfile_from_path(file_path: str, filename: str, max_lines: int = 10000) -> UploadFile:
    with open(file_path, "rb") as f:
        # Read only the first max_lines lines
        lines = []
        for i, line in enumerate(f):
            if i >= max_lines:
                break
            lines.append(line)
        file_content = b"".join(lines)
        file_stream = io.BytesIO(file_content)
    return UploadFile(
        filename=filename,
        file=file_stream
    )

In [3]:
class ConvertToDf():

    @staticmethod
    def read_file(file: UploadFile) -> pl.DataFrame:
        """Read PGN file and convert to Polars DataFrame."""
        
        # Read file content
        content = file.file.read().decode('utf-8')
        
        # Parse PGN text
        games = ConvertToDf._parse_pgn(content)
        
        df = pl.DataFrame(games)

        df = ConvertToDf._apply_schema(df)

        df = ConvertToDf._add_move_count(df)
        
        return df
    
    @staticmethod
    def _parse_pgn(pgn_text: str) -> list[dict[str, str]]:
        """Parse PGN text into game dictionaries."""
        games: list[dict[str, str]] = []
        game_blocks = re.split(r'\n\n(?=\[Event)', pgn_text.strip())
        
        for block in game_blocks:
            if not block.strip():
                continue
            
            game_data = ConvertToDf._parse_game_block(block)
            if game_data:
                games.append(game_data)
        
        return games
    
    @staticmethod
    def _parse_game_block(block: str) -> Optional[dict[str, str]]:
        """Parse a single game block."""
        game_data: dict[str, str] = {}
        
        # Extract headers
        headers = re.findall(r'\[(\w+)\s+"([^"]*)"\]', block)
        for key, value in headers:
            game_data[key] = value
        
        # Extract moves
        moves_match = re.search(
            r'\]\n\n(.+?)(?:\s+(?:1-0|0-1|1/2-1/2|\*))?$',
            block,
            re.DOTALL
        )
        
        if moves_match:
            moves_text = moves_match.group(1).strip()
            # Remove comments in curly braces
            moves_clean = re.sub(r'\{[^}]*\}', '', moves_text)
            # Normalize whitespace
            moves_clean = re.sub(r'\s+', ' ', moves_clean).strip()
            game_data['Moves'] = moves_clean
        else:
            game_data['Moves'] = ''
        
        return game_data if game_data else None
    
    @staticmethod
    def _apply_schema(df: pl.DataFrame) -> pl.DataFrame:
        df = df.with_columns(pl.col('WhiteElo').cast(pl.Int16),
                             pl.col('BlackElo').cast(pl.Int16))
        return df
    
    @staticmethod
    def _count_moves(moves: str) -> int:
        """Count the number of full moves in a game."""
        game = pgn.read_game(io.StringIO(moves))
        if game is None:
           return 0
        return sum(1 for _ in game.mainline_moves()) // 2
    
    @staticmethod
    def _add_move_count(df: pl.DataFrame) -> pl.DataFrame:
        """Add move count column to dataframe."""
        return df.with_columns(
            pl.col('Moves')
            .map_elements(ConvertToDf._count_moves, return_dtype=pl.Int16)
            .alias('NumMoves')
        )
        


In [4]:
@dataclass
class FilterConfig:
    terminations: Sequence[str] = field(
        default_factory=lambda: ("Normal")
    )
    events: Sequence[str] = field(
        default_factory=lambda: (
            "Rated Blitz game",
            "Rated Rapid game",
        )
    )
    elo_range: tuple[int, int] = field(default_factory=lambda: (1400, 2800))
    min_moves: int = 15

filter_config = FilterConfig()


In [5]:
class GameFilter():

    @staticmethod
    def apply_filters(df: pl.DataFrame, config:FilterConfig) -> pl.DataFrame:
        df = GameFilter._filter_by_items_in_list(df, config.terminations,'Termination')
        df = GameFilter._filter_by_items_in_list(df, config.events, 'Event')
        df = GameFilter._filter_by_elo(df, config.elo_range)
        df = GameFilter._filter_by_move_count(df, config.min_moves)
        return df

    @staticmethod
    def _filter_by_elo(df: pl.DataFrame,
                    range: tuple[int, int]) -> pl.DataFrame:

        return df.filter(pl.col('WhiteElo').is_between(range[0], range[1]))

    @staticmethod
    def _filter_by_items_in_list(df: pl.DataFrame, 
                            list:Sequence[str], target_col: str) -> pl.DataFrame:
        pattern = '|'.join(list)
        return df.filter(pl.col(target_col).str.contains(pattern))

    @staticmethod
    def _filter_by_move_count(df: pl.DataFrame,
                           min_moves: int) -> pl.DataFrame:
        return df.filter(pl.col('NumMoves')>=min_moves)

In [6]:
# INPUT = '../../../data/sample_data.pgn'
INPUT_BIG = '/home/ge/MCD/Tesis/Tesis/chess/lichess_db_standard_rated_2025-07_split1.pgn'
# file = create_uploadfile_from_path(INPUT, 'sample_data.pgn')
file = create_uploadfile_from_path(INPUT_BIG, 'lichess_db_standard_rated_2025-07_split1.pgn', max_lines=40*100000)

In [7]:
df = ConvertToDf.read_file(file)
df_filtered = GameFilter.apply_filters(df, filter_config)

In [8]:
df_filtered['Result'].value_counts()

Result,count
str,u32
"""0-1""",31442
"""1/2-1/2""",3206
"""1-0""",33228


In [9]:
df_filtered.shape

(67876, 20)

In [10]:
@dataclass
class MappingConfig:
    result_map: dict[str, int] = field(
        default_factory=lambda: {
            "1-0": 1,
            "0-1": -1,
            "1/2-1/2": 0,
        }
    )
    event_map: dict[str, str] = field(
        default_factory=lambda: {
            "Rated Blitz game": "Blitz",
            "Rated Bullet game": "Bullet",
            "Rated Rapid game": "Rapid",
        }
    )

mapping_config = MappingConfig()


In [11]:
class GameMapping():

    @staticmethod
    def apply_mappings(df: pl.DataFrame, config:MappingConfig) -> pl.DataFrame:
        df = GameMapping._map_result(df, config.result_map)
        df = GameMapping._map_event(df, config.event_map)
        return df

    @staticmethod
    def _map_result(df: pl.DataFrame,
                    mapping: dict[str, int]) -> pl.DataFrame:

        return df.with_columns(
            pl.col('Result').map_elements(lambda x: mapping.get(x, None), return_dtype=pl.Int8)
        )
    
    @staticmethod
    def _map_event(df: pl.DataFrame,
                   mapping: dict[str, str]) -> pl.DataFrame:

        return df.with_columns(
            pl.col('Event').map_elements(lambda x: mapping.get(x, None), return_dtype=pl.Utf8)
        )

In [12]:
df_mapped = GameMapping.apply_mappings(df_filtered, mapping_config)

In [13]:
df_mapped['Result'].value_counts()

Result,count
i8,u32
-1,31442
1,33228
0,3206


In [14]:
@dataclass
class FeatureEngineerConfig:
    """Configuration for feature engineering."""
    move_number: int = 15  # Which move to analyze (full move number)
    piece_values: dict[int, int] = field(
        default_factory=lambda: {
            chess.PAWN: 1,
            chess.KNIGHT: 3,
            chess.BISHOP: 3,
            chess.ROOK: 5,
            chess.QUEEN: 9,
            chess.KING: 0
        }
    )
    center_squares: list[int] = field(
        default_factory=lambda: [chess.D4, chess.E4, chess.D5, chess.E5]
    )
    extended_center_squares: list[int] = field(
        default_factory=lambda: [
            chess.C3, chess.D3, chess.E3, chess.F3,
            chess.C4, chess.D4, chess.E4, chess.F4,
            chess.C5, chess.D5, chess.E5, chess.F5,
            chess.C6, chess.D6, chess.E6, chess.F6
        ]
    )



In [15]:
class FeatureEngineer:
    @staticmethod
    def add_features(df: pl.DataFrame, config: FeatureEngineerConfig) -> pl.DataFrame:
        """Add all chess features to the dataframe."""
        features_list: list[dict[str, float]] = []

        for moves in df["Moves"]:
            board = FeatureEngineer._get_board_at_move(moves, config.move_number)

            features: dict[str, float] = {}
            features.update(FeatureEngineer._calculate_material(board, config))
            features.update(FeatureEngineer._calculate_attacked_pieces(board, config))
            features.update(FeatureEngineer._calculate_center_control(board, config))
            features.update(FeatureEngineer._calculate_castling_rights(board))
            features.update(FeatureEngineer._calculate_mobility(board, config))
            
            # Non-overlapping new features only
            features.update(FeatureEngineer._calculate_opponent_aggression(board))
            features.update(FeatureEngineer._calculate_piece_positions(board))
            features.update(FeatureEngineer._calculate_dof_x_material(board, config))

            features_list.append(features)

        features_df = pl.DataFrame(features_list)
        return pl.concat([df, features_df], how="horizontal")

    @staticmethod
    def _get_board_at_move(moves: str, move_number: int) -> Optional[chess.Board]:
        """Get board state at a specific move number."""
        try:
            game = pgn.read_game(io.StringIO(moves))
            if game is None:
                return None

            board = game.board()
            move_count = 0

            for move in game.mainline_moves():
                board.push(move)
                move_count += 1
                if move_count == move_number * 2:
                    return board

            return board if move_count > 0 else None
        except:
            return None

    @staticmethod
    def _calculate_material(
        board: Optional[chess.Board], config: FeatureEngineerConfig
    ) -> dict[str, int]:
        """Calculate material balance on the board."""
        white_material = 0
        black_material = 0

        if board is None:
            return {"white_material": 0, "black_material": 0, "material_diff": 0}
        
        for square in chess.SQUARES:
            piece = board.piece_at(square)
            if piece:
                value = config.piece_values[piece.piece_type]
                if piece.color == chess.WHITE:
                    white_material += value
                else:
                    black_material += value

        return {
            "white_material": white_material,
            "black_material": black_material,
            "material_diff": white_material - black_material,
        }

    @staticmethod
    def _calculate_attacked_pieces(
        board: Optional[chess.Board], config: FeatureEngineerConfig
    ) -> dict[str, int]:
        """Calculate number and value of pieces under attack."""
        if board is None:
            return {
                "white_pieces_attacked": 0,
                "white_attacked_value": 0,
                "black_pieces_attacked": 0,
                "black_attacked_value": 0,
                "attacked_diff": 0,
            }

        white_attacked_count = 0
        white_attacked_value = 0
        black_attacked_count = 0
        black_attacked_value = 0

        for square in chess.SQUARES:
            piece = board.piece_at(square)
            if piece:
                is_attacked = board.is_attacked_by(not piece.color, square)

                if is_attacked:
                    piece_value = config.piece_values[piece.piece_type]

                    if piece.color == chess.WHITE:
                        white_attacked_count += 1
                        white_attacked_value += piece_value
                    else:
                        black_attacked_count += 1
                        black_attacked_value += piece_value

        return {
            "white_pieces_attacked": white_attacked_count,
            "white_attacked_value": white_attacked_value,
            "black_pieces_attacked": black_attacked_count,
            "black_attacked_value": black_attacked_value,
            "attacked_diff": black_attacked_value - white_attacked_value,
        }

    @staticmethod
    def _calculate_center_control(
        board: Optional[chess.Board], config: FeatureEngineerConfig
    ) -> dict[str, int]:
        """Calculate center control metrics."""
        if board is None:
            return {
                "white_center_pieces": 0,
                "black_center_pieces": 0,
                "white_center_control": 0,
                "black_center_control": 0,
                "center_control_diff": 0,
                "white_extended_control": 0,
                "black_extended_control": 0,
                "extended_control_diff": 0,
            }

        white_center_pieces = 0
        black_center_pieces = 0

        for square in config.center_squares:
            piece = board.piece_at(square)
            if piece:
                if piece.color == chess.WHITE:
                    white_center_pieces += 1
                else:
                    black_center_pieces += 1

        white_center_attacks = sum(
            1 for sq in config.center_squares if board.is_attacked_by(chess.WHITE, sq)
        )
        black_center_attacks = sum(
            1 for sq in config.center_squares if board.is_attacked_by(chess.BLACK, sq)
        )

        white_extended_attacks = sum(
            1
            for sq in config.extended_center_squares
            if board.is_attacked_by(chess.WHITE, sq)
        )
        black_extended_attacks = sum(
            1
            for sq in config.extended_center_squares
            if board.is_attacked_by(chess.BLACK, sq)
        )

        return {
            "white_center_pieces": white_center_pieces,
            "black_center_pieces": black_center_pieces,
            "white_center_control": white_center_attacks,
            "black_center_control": black_center_attacks,
            "center_control_diff": white_center_attacks - black_center_attacks,
            "white_extended_control": white_extended_attacks,
            "black_extended_control": black_extended_attacks,
            "extended_control_diff": white_extended_attacks - black_extended_attacks,
        }

    @staticmethod
    def _calculate_castling_rights(board: Optional[chess.Board]) -> dict[str, int]:
        """Calculate castling rights status."""
        if board is None:
            return {
                "white_can_castle_kingside": 0,
                "white_can_castle_queenside": 0,
                "black_can_castle_kingside": 0,
                "black_can_castle_queenside": 0,
                "white_has_castled": 0,
                "black_has_castled": 0,
                "white_castling_rights_count": 0,
                "black_castling_rights_count": 0,
            }

        white_king_sq = board.king(chess.WHITE)
        black_king_sq = board.king(chess.BLACK)

        white_has_castled = (
            not board.has_castling_rights(chess.WHITE)
            and white_king_sq is not None
            and white_king_sq in [chess.G1, chess.C1]
        )

        black_has_castled = (
            not board.has_castling_rights(chess.BLACK)
            and black_king_sq is not None
            and black_king_sq in [chess.G8, chess.C8]
        )

        return {
            "white_can_castle_kingside": int(
                board.has_kingside_castling_rights(chess.WHITE)
            ),
            "white_can_castle_queenside": int(
                board.has_queenside_castling_rights(chess.WHITE)
            ),
            "black_can_castle_kingside": int(
                board.has_kingside_castling_rights(chess.BLACK)
            ),
            "black_can_castle_queenside": int(
                board.has_queenside_castling_rights(chess.BLACK)
            ),
            "white_has_castled": int(white_has_castled),
            "black_has_castled": int(black_has_castled),
            "white_castling_rights_count": (
                board.has_kingside_castling_rights(chess.WHITE)
                + board.has_queenside_castling_rights(chess.WHITE)
            ),
            "black_castling_rights_count": (
                board.has_kingside_castling_rights(chess.BLACK)
                + board.has_queenside_castling_rights(chess.BLACK)
            ),
        }

    @staticmethod
    def _calculate_mobility(
        board: Optional[chess.Board], config: FeatureEngineerConfig
    ) -> dict[str, float]:
        """Calculate mobility (degrees of freedom) metrics."""
        if board is None:
            return {
                "white_mobility": 0,
                "black_mobility": 0,
                "mobility_diff": 0,
                "white_weighted_mobility": 0.0,
                "black_weighted_mobility": 0.0,
                "weighted_mobility_diff": 0.0,
            }

        current_turn = board.turn

        if current_turn == chess.WHITE:
            white_legal_moves = board.legal_moves.count()
            board.turn = chess.BLACK
            black_legal_moves = board.legal_moves.count()
            board.turn = chess.WHITE
        else:
            black_legal_moves = board.legal_moves.count()
            board.turn = chess.WHITE
            white_legal_moves = board.legal_moves.count()
            board.turn = chess.BLACK

        white_weighted = FeatureEngineer._calculate_weighted_mobility(
            board, chess.WHITE, config
        )
        black_weighted = FeatureEngineer._calculate_weighted_mobility(
            board, chess.BLACK, config
        )

        return {
            "white_mobility": white_legal_moves,
            "black_mobility": black_legal_moves,
            "mobility_diff": white_legal_moves - black_legal_moves,
            "white_weighted_mobility": white_weighted,
            "black_weighted_mobility": black_weighted,
            "weighted_mobility_diff": white_weighted - black_weighted,
        }

    @staticmethod
    def _calculate_weighted_mobility(
        board: chess.Board, color: bool, config: FeatureEngineerConfig
    ) -> float:
        """Calculate weighted mobility (sum of legal moves × piece value)."""
        weighted_sum = 0.0

        for square in chess.SQUARES:
            piece = board.piece_at(square)
            if piece and piece.color == color:
                piece_value = config.piece_values[piece.piece_type]
                moves_from_square = sum(
                    1 for move in board.legal_moves if move.from_square == square
                )
                weighted_sum += moves_from_square * piece_value

        return weighted_sum

    # NEW FEATURES FROM COMMENTED CODE

    @staticmethod
    def _calculate_position_advantage(board: Optional[chess.Board]) -> dict[str, int]:
        """Calculate advantage based on number of squares under attack by each team."""
        if board is None:
            return {"position_advantage": 0}
        
        w = len([x for x in range(64) if board.is_attacked_by(chess.WHITE, chess.SQUARES[x])])
        b = len([x for x in range(64) if board.is_attacked_by(chess.BLACK, chess.SQUARES[x])])
        return {"position_advantage": w - b}

    @staticmethod
    def _calculate_center_advantage(board: Optional[chess.Board]) -> dict[str, int]:
        """Calculate advantage based on squares under attack in center."""
        if board is None:
            return {"center_advantage": 0}
        
        center = [18, 19, 20, 21, 26, 27, 28, 29, 34, 35, 36, 37, 42, 43, 44, 45]
        w = len([x for x in range(64) if board.is_attacked_by(chess.WHITE, chess.SQUARES[x]) and x in center])
        b = len([x for x in range(64) if board.is_attacked_by(chess.BLACK, chess.SQUARES[x]) and x in center])
        return {"center_advantage": w - b}

    @staticmethod
    def _calculate_opponent_aggression(board: Optional[chess.Board]) -> dict[str, int]:
        """Calculate advantage based on squares under attack on opponent half."""
        if board is None:
            return {"opponent_aggression": 0}
        
        w = len([x for x in range(64) if board.is_attacked_by(chess.WHITE, chess.SQUARES[x]) and x > 31])
        b = len([x for x in range(64) if board.is_attacked_by(chess.BLACK, chess.SQUARES[x]) and x < 31])
        return {"opponent_aggression": w - b}

    @staticmethod
    def _calculate_degrees_of_freedom(board: Optional[chess.Board]) -> dict[str, int]:
        """Calculate degrees of freedom (legally accessible squares)."""
        if board is None:
            return {"degrees_of_freedom": 0}
        
        try:
            dof1 = len(list(board.legal_moves)) if board.turn else -len(list(board.legal_moves))
            board.push(chess.Move.null())
            dof2 = len(list(board.legal_moves)) if board.turn else -len(list(board.legal_moves))
            board.pop()
            return {"degrees_of_freedom": dof1 + dof2}
        except:
            return {"degrees_of_freedom": 0}

    @staticmethod
    def _calculate_aggression(board: Optional[chess.Board]) -> dict[str, int]:
        """Calculate advantage based on number of opponent pieces under attack."""
        if board is None:
            return {"aggression": 0}
        
        w = len([x for x in range(64) 
                if board.is_attacked_by(chess.WHITE, chess.SQUARES[x]) 
                and x in list(chess.scan_reversed(board.occupied_co[chess.BLACK]))])
        b = len([x for x in range(64) 
                if board.is_attacked_by(chess.BLACK, chess.SQUARES[x]) 
                and x in list(chess.scan_reversed(board.occupied_co[chess.WHITE]))])
        return {"aggression": w - b}

    @staticmethod
    def _calculate_piece_positions(board: Optional[chess.Board]) -> dict[str, int]:
        """Calculate piece positioning scores for each piece type."""
        if board is None:
            return {
                "queen_position": 0,
                "knight_position": 0,
                "bishop_position": 0,
                "rook_position": 0,
                "pawn_position": 0,
            }
        
        def piece_position_score(piece_type: int) -> int:
            w = [idx for idx, sq in enumerate(chess.SQUARES) 
                 if board.piece_at(sq) and board.piece_at(sq).piece_type == piece_type 
                 and board.piece_at(sq).color == chess.WHITE]
            b = [idx for idx, sq in enumerate(chess.SQUARES) 
                 if board.piece_at(sq) and board.piece_at(sq).piece_type == piece_type 
                 and board.piece_at(sq).color == chess.BLACK]
            
            w_attacks = sum(len(list(board.attacks(chess.SQUARES[ind]))) for ind in w)
            b_attacks = sum(len(list(board.attacks(chess.SQUARES[ind]))) for ind in b)
            return w_attacks - b_attacks
        
        return {
            "queen_position": piece_position_score(chess.QUEEN),
            "knight_position": piece_position_score(chess.KNIGHT),
            "bishop_position": piece_position_score(chess.BISHOP),
            "rook_position": piece_position_score(chess.ROOK),
            "pawn_position": piece_position_score(chess.PAWN),
        }

    @staticmethod
    def _calculate_pawn_structure(board: Optional[chess.Board]) -> dict[str, int]:
        """Calculate pawn structure score based on protection."""
        if board is None:
            return {"pawn_structure": 0}
        
        pawn_w = [idx for idx, sq in enumerate(chess.SQUARES) 
                  if board.piece_at(sq) and board.piece_at(sq).piece_type == chess.PAWN 
                  and board.piece_at(sq).color == chess.WHITE]
        pawn_b = [idx for idx, sq in enumerate(chess.SQUARES) 
                  if board.piece_at(sq) and board.piece_at(sq).piece_type == chess.PAWN 
                  and board.piece_at(sq).color == chess.BLACK]
        
        w = len([x for x in range(64) if board.is_attacked_by(chess.WHITE, chess.SQUARES[x]) and x in pawn_w])
        b = len([x for x in range(64) if board.is_attacked_by(chess.BLACK, chess.SQUARES[x]) and x in pawn_b])
        return {"pawn_structure": w - b}

    @staticmethod
    def _calculate_pieces_protected(board: Optional[chess.Board]) -> dict[str, int]:
        """Calculate score based on pieces being protected."""
        if board is None:
            return {"pieces_protected": 0}
        
        all_w = [idx for idx, sq in enumerate(chess.SQUARES) 
                 if board.piece_at(sq) and board.piece_at(sq).color == chess.WHITE]
        all_b = [idx for idx, sq in enumerate(chess.SQUARES) 
                 if board.piece_at(sq) and board.piece_at(sq).color == chess.BLACK]
        
        w = len([x for x in range(64) if board.is_attacked_by(chess.WHITE, chess.SQUARES[x]) and x in all_w])
        b = len([x for x in range(64) if board.is_attacked_by(chess.BLACK, chess.SQUARES[x]) and x in all_b])
        return {"pieces_protected": w - b}

    @staticmethod
    def _calculate_dof_x_material(
        board: Optional[chess.Board], config: FeatureEngineerConfig
    ) -> dict[str, float]:
        """Calculate degrees of freedom multiplied by material score."""
        if board is None:
            return {"dof_x_material": 0.0}
        
        try:
            scorew = sum(config.piece_values[p.piece_type] 
                        for sq in chess.SQUARES 
                        if (p := board.piece_at(sq)) and p.color == chess.WHITE)
            scoreb = sum(config.piece_values[p.piece_type] 
                        for sq in chess.SQUARES 
                        if (p := board.piece_at(sq)) and p.color == chess.BLACK)
            
            dof1 = len(list(board.legal_moves)) if board.turn else -len(list(board.legal_moves))
            board.push(chess.Move.null())
            dof2 = len(list(board.legal_moves)) if board.turn else -len(list(board.legal_moves))
            board.pop()
            
            scorew2 = dof1 * scorew if dof1 > 0 else dof2 * scorew
            scoreb2 = dof1 * scoreb if dof1 < 0 else dof2 * scoreb
            
            return {"dof_x_material": float(scorew2 - scoreb2)}
        except:
            return {"dof_x_material": 0.0}

In [16]:
df_features = FeatureEngineer.add_features(df_mapped, FeatureEngineerConfig())

In [17]:
df.columns

['Event',
 'Site',
 'Date',
 'Round',
 'White',
 'Black',
 'Result',
 'UTCDate',
 'UTCTime',
 'WhiteElo',
 'BlackElo',
 'WhiteRatingDiff',
 'BlackRatingDiff',
 'ECO',
 'Opening',
 'TimeControl',
 'Termination',
 'Moves',
 'BlackTitle',
 'NumMoves']

In [18]:
df_features.sample()

Event,Site,Date,Round,White,Black,Result,UTCDate,UTCTime,WhiteElo,BlackElo,WhiteRatingDiff,BlackRatingDiff,ECO,Opening,TimeControl,Termination,Moves,BlackTitle,NumMoves,white_material,black_material,material_diff,white_pieces_attacked,white_attacked_value,black_pieces_attacked,black_attacked_value,attacked_diff,white_center_pieces,black_center_pieces,white_center_control,black_center_control,center_control_diff,white_extended_control,black_extended_control,extended_control_diff,white_can_castle_kingside,white_can_castle_queenside,black_can_castle_kingside,black_can_castle_queenside,white_has_castled,black_has_castled,white_castling_rights_count,black_castling_rights_count,white_mobility,black_mobility,mobility_diff,white_weighted_mobility,black_weighted_mobility,weighted_mobility_diff,opponent_aggression,queen_position,knight_position,bishop_position,rook_position,pawn_position,dof_x_material
str,str,str,str,str,str,i8,str,str,i16,i16,str,str,str,str,str,str,str,str,i16,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,i64,i64,i64,i64,i64,i64,f64
"""Blitz""","""https://lichess.org/FF2wVEnz""","""2025.07.01""","""-""","""Slick_Stick""","""TolBrandir""",-1,"""2025.07.01""","""02:37:25""",1756,1845,"""-4""","""+5""","""A05""","""King's Indian Attack""","""180+2""","""Time forfeit""","""1. Nf3 1... d5 2. g3 2... Nf6 …",null,30,35,35,0,4,6,1,1,-5,1,0,3,4,-1,9,14,-5,0,0,0,0,0,1,0,0,29,49,-20,111.0,0.0,111.0,-11,-5,-10,1,-9,0,2730.0


In [19]:
@dataclass
class SelectionConfig:
    features_to_drop: Sequence[str] = field(default_factory=lambda: [
        'Site',
        'Date',
        'Round',
        'White',
        'Black',
        'UTCDate',
        'UTCTime',
        'BlackTitle',
        'NumMoves',
        'Moves',
        'WhiteRatingDiff',
        'BlackRatingDiff',
        
    ])

In [20]:
class FeatureSelector():
    
    @staticmethod
    def select_features(df: pl.DataFrame, config:SelectionConfig) -> pl.DataFrame:
        return df.drop(config.features_to_drop)

In [21]:
df_selected = FeatureSelector.select_features(df_features, SelectionConfig())
df_selected.sample(5)

Event,Result,WhiteElo,BlackElo,ECO,Opening,TimeControl,Termination,white_material,black_material,material_diff,white_pieces_attacked,white_attacked_value,black_pieces_attacked,black_attacked_value,attacked_diff,white_center_pieces,black_center_pieces,white_center_control,black_center_control,center_control_diff,white_extended_control,black_extended_control,extended_control_diff,white_can_castle_kingside,white_can_castle_queenside,black_can_castle_kingside,black_can_castle_queenside,white_has_castled,black_has_castled,white_castling_rights_count,black_castling_rights_count,white_mobility,black_mobility,mobility_diff,white_weighted_mobility,black_weighted_mobility,weighted_mobility_diff,opponent_aggression,queen_position,knight_position,bishop_position,rook_position,pawn_position,dof_x_material
str,i8,i16,i16,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,i64,i64,i64,i64,i64,i64,f64
"""Blitz""",1,2114,2102,"""B33""","""Sicilian Defense: Lasker-Pelik…","""180+0""","""Normal""",33,34,-1,1,5,2,2,-3,2,0,2,2,0,10,10,0,0,0,0,0,1,1,0,0,43,37,6,165.0,0.0,165.0,2,-1,8,-7,5,-2,2677.0
"""Blitz""",1,1961,1951,"""B00""","""Pirc Defense""","""300+3""","""Normal""",34,34,0,2,4,2,2,-2,0,1,2,3,-1,9,11,-2,0,0,0,0,1,1,0,0,40,38,2,197.0,0.0,197.0,3,4,0,4,2,0,2652.0
"""Blitz""",-1,1824,1840,"""D10""","""Slav Defense""","""180+0""","""Normal""",29,29,0,2,2,3,5,3,2,0,4,2,2,13,10,3,0,0,1,1,1,0,0,2,41,43,-2,233.0,0.0,233.0,2,4,0,0,1,0,2436.0
"""Rapid""",-1,1797,1829,"""C41""","""Philidor Defense""","""600+5""","""Normal""",36,35,1,4,5,1,1,-4,0,2,2,4,-2,8,14,-6,0,0,0,0,1,1,0,0,3,44,-41,5.0,0.0,5.0,-7,-6,-6,2,-4,-4,1648.0
"""Blitz""",1,1995,1993,"""D06""","""Queen's Gambit Declined: Marsh…","""300+0""","""Normal""",38,38,0,1,1,1,1,0,3,1,2,2,0,11,10,1,0,0,0,0,1,1,0,0,36,37,-1,146.0,0.0,146.0,2,0,2,5,0,0,2774.0


In [22]:
df_selected.schema

Schema([('Event', String),
        ('Result', Int8),
        ('WhiteElo', Int16),
        ('BlackElo', Int16),
        ('ECO', String),
        ('Opening', String),
        ('TimeControl', String),
        ('Termination', String),
        ('white_material', Int64),
        ('black_material', Int64),
        ('material_diff', Int64),
        ('white_pieces_attacked', Int64),
        ('white_attacked_value', Int64),
        ('black_pieces_attacked', Int64),
        ('black_attacked_value', Int64),
        ('attacked_diff', Int64),
        ('white_center_pieces', Int64),
        ('black_center_pieces', Int64),
        ('white_center_control', Int64),
        ('black_center_control', Int64),
        ('center_control_diff', Int64),
        ('white_extended_control', Int64),
        ('black_extended_control', Int64),
        ('extended_control_diff', Int64),
        ('white_can_castle_kingside', Int64),
        ('white_can_castle_queenside', Int64),
        ('black_can_castle_kingside', Int6

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import matthews_corrcoef, confusion_matrix

from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [24]:
X = df_selected.drop('Result')
y = df_selected['Result']

In [25]:
X.shape

(67876, 44)

In [52]:
ohe_cols = ['Event','TimeControl', 'Termination']

ordinal_cols = ['ECO','Opening']

numeric_cols = list(set(X.columns) - set(ohe_cols) - set(ordinal_cols))

from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array([-1, 0, 1])
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))

In [53]:
preprecessing_pipeline = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('ord', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ohe_cols)
])

logistic_model = Pipeline(steps=[
    ('preprocessor', preprecessing_pipeline),
    ('classifier', LogisticRegression(max_iter=1000, class_weight=class_weights, random_state=42))
])

random_forest_model = Pipeline(steps=[
    ('preprocessor', preprecessing_pipeline),
    ('classifier', RandomForestClassifier(n_estimators=100,max_depth=10,min_samples_split=80,max_samples=0.5, random_state=42, class_weight=class_weights))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [54]:
logistic_model.fit(X_train, y_train)

y_pred_log = logistic_model.predict(X_test)

metrics_log = {
    'matthews_corrcoef': matthews_corrcoef(y_test, y_pred_log),
    'confusion_matrix': confusion_matrix(y_test, y_pred_log) }

metrics_log

{'matthews_corrcoef': 0.10670236588840515,
 'confusion_matrix': array([[2485, 1937, 1867],
        [ 161,  294,  186],
        [1879, 2034, 2733]])}

In [55]:
random_forest_model.fit(X_train, y_train)

y_pred_rf = random_forest_model.predict(X_test)

metrics_rf = {
    'matthews_corrcoef': matthews_corrcoef(y_test, y_pred_rf),
    'confusion_matrix': confusion_matrix(y_test, y_pred_rf) }

metrics_rf

{'matthews_corrcoef': 0.11250354411305044,
 'confusion_matrix': array([[3149, 1179, 1961],
        [ 236,  184,  221],
        [2533, 1188, 2925]])}

In [56]:
# Get feature names after preprocessing
feature_names = random_forest_model.named_steps['preprocessor'].get_feature_names_out()

# Random Forest importances
rf_importances = random_forest_model.named_steps['classifier'].feature_importances_
df_rf_importances = pl.DataFrame({'feature': feature_names, 'importance': rf_importances}).sort('importance', descending=True)

# Logistic Regression importances (absolute value of coefficients)
log_coeffs = logistic_model.named_steps['classifier'].coef_[0]
df_log_importances = pl.DataFrame({'feature': feature_names, 'importance': abs(log_coeffs)}).sort('importance', descending=True)

df_rf_importances.head(10)

feature,importance
str,f64
"""num__material_diff""",0.064912
"""num__WhiteElo""",0.050372
"""num__BlackElo""",0.045769
"""num__dof_x_material""",0.045352
"""num__white_material""",0.043207
"""num__mobility_diff""",0.037693
"""num__white_weighted_mobility""",0.03641
"""num__weighted_mobility_diff""",0.034437
"""num__black_material""",0.034252


In [57]:
df_log_importances.head(10)

feature,importance
str,f64
"""num__WhiteElo""",0.231149
"""num__material_diff""",0.184104
"""ohe__Termination_Time forfeit""",0.176219
"""num__BlackElo""",0.169014
"""ohe__Termination_Normal""",0.118024
"""num__black_material""",0.094766
"""ohe__TimeControl_180+2""",0.091815
"""num__black_pieces_attacked""",0.087296
"""num__black_attacked_value""",0.086064


In [58]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_log, target_names=['Black Win', 'Draw', 'White Win']))
print(classification_report(y_test, y_pred_rf, target_names=['Black Win', 'Draw', 'White Win']))

              precision    recall  f1-score   support

   Black Win       0.55      0.40      0.46      6289
        Draw       0.07      0.46      0.12       641
   White Win       0.57      0.41      0.48      6646

    accuracy                           0.41     13576
   macro avg       0.40      0.42      0.35     13576
weighted avg       0.54      0.41      0.45     13576

              precision    recall  f1-score   support

   Black Win       0.53      0.50      0.52      6289
        Draw       0.07      0.29      0.12       641
   White Win       0.57      0.44      0.50      6646

    accuracy                           0.46     13576
   macro avg       0.39      0.41      0.38     13576
weighted avg       0.53      0.46      0.49     13576



In [59]:
from lightgbm import LGBMClassifier

In [60]:
lightgbm_model = Pipeline(steps=[
    ('preprocessor', preprecessing_pipeline),
    ('classifier', LGBMClassifier(n_estimators=300, learning_rate=0.01, max_depth=10, verbose=-1, random_state=42, class_weight=class_weights))
])

lightgbm_model.fit(X_train, y_train)
y_pred_lgbm = lightgbm_model.predict(X_test)
metrics_lgbm = {
    'matthews_corrcoef': matthews_corrcoef(y_test, y_pred_lgbm),
    'confusion_matrix': confusion_matrix(y_test, y_pred_lgbm) }
metrics_lgbm


{'matthews_corrcoef': 0.12257355064009465,
 'confusion_matrix': array([[2910, 1692, 1687],
        [ 206,  267,  168],
        [2202, 1766, 2678]])}

In [65]:
print(classification_report(y_test, y_pred_lgbm, target_names=['Black Win', 'Draw', 'White Win']))

              precision    recall  f1-score   support

   Black Win       0.55      0.46      0.50      6289
        Draw       0.07      0.42      0.12       641
   White Win       0.59      0.40      0.48      6646

    accuracy                           0.43     13576
   macro avg       0.40      0.43      0.37     13576
weighted avg       0.55      0.43      0.47     13576



In [61]:
feaature_names = lightgbm_model.named_steps['preprocessor'].get_feature_names_out()
lgbm_importances = lightgbm_model.named_steps['classifier'].feature_importances_
df_lgbm_importances = pl.DataFrame({'feature': feature_names, 'importance': lgbm_importances}).sort('importance', descending=True)
df_lgbm_importances.head(10)

feature,importance
str,i32
"""num__BlackElo""",1934
"""num__WhiteElo""",1858
"""num__material_diff""",1737
"""num__mobility_diff""",1348
"""num__dof_x_material""",1186
"""ord__ECO""",1129
"""num__weighted_mobility_diff""",1095
"""ord__Opening""",1029
"""num__black_mobility""",1025
